# Bước 5 Làm sạch text Hán Việt
Chế độ mặc định `parts` đọc TXT OCR từng ảnh trong `02_ocr_tho` và ráp các phần cùng bài. Dịch nghĩa Bắc Sứ lấy từ hai TXT trong `01_nguon`.
Chạy CPU. Phần hiển thị ảnh dùng Pillow, và chế độ toàn trang PDF dùng pypdfium2.

**Quy tắc đã thống nhất:** giữ số có nghĩa; chữ dính số chỉ gợi ý, sửa sau khi duyệt.
Không xóa từ ngoài từ điển, không tự sửa bản dịch, không đổi Hán phồn thể/giản thể.
Không dùng stemming, stopword removal hoặc BoW để xóa nội dung corpus.

Mặc định chạy dữ liệu thật trên local và xuất bảng duyệt; text sạch chỉ xuất sau khi bạn duyệt.
Notebook không chứng nhận hoàn thành bước 5
nếu chưa đối chiếu ảnh, xử lý đầu/chân trang, chú thích, bảng và các chỗ ngắt trang.

Luồng duyệt: làm sạch Hán/Việt → xem ảnh theo trang → yêu cầu Codex đối chiếu và cập nhật CSV → kiểm tra quyết định → xuất hàng loạt.


## Đối chiếu tài liệu
`Guideline.pdf`, trang 3, bước 5 yêu cầu: bỏ đầu/chân trang và nội dung ngoài phần chính;
ghép chỗ ngắt trang; loại OCR lặp vùng; bỏ chú thích ngoài nội dung Hán và chỉ mục [1], (1);
sửa rác/khoảng trắng; chuẩn hóa dấu câu không đổi nghĩa; **giữ tên riêng, niên hiệu, con số,
đơn vị đo theo nguồn**; bỏ bảng và tranh.

Notebook tự xử lý khoảng trắng và Unicode Latin có dấu. Các thao tác có thể mất nội dung
được đưa vào bảng duyệt theo dòng. Số trang, tiêu đề và chú thích phải xác nhận bằng ảnh.
Dòng lặp có thể là điệp ngữ thơ nên chỉ cảnh báo. Ranh giới trang `\f` được giữ để truy vết.
Chưa tự tách câu hay dóng hàng: đó là bước 6–7.


## 1 Cấu hình lượt chạy
Chọn sách bằng `SELECTED_CODES` và bản văn bằng `SELECTED_KINDS`.
Lượt chạy đầu tạo bảng duyệt. Bật ghi TXT sau khi đã xem lại các quyết định ở mục 5.


In [ ]:
# 1. Thiết lập nguồn, sách và phiên bản
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from zipfile import ZipFile
from runpy import run_path
import csv
import hashlib
import io
import json
import re
import unicodedata
from datetime import datetime, timezone

here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (here, *here.parents) if (p / "ten_file.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open notebook inside the repository containing ten_file.py.")
TOPIC_CODE = run_path(str(PROJECT_ROOT / "ten_file.py"))["MA_DE_TAI"]
BOOK_FOLDERS = {"CPO_09": "BacSu", "CPO_11": "ThiDinh", "CPO_12": "LyTran"}
INPUT_PATH = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "san_pham" / "TV04"

SELECTED_CODES = {"CPO_09", "CPO_11", "CPO_12"}
SELECTED_KINDS = {"han", "dich_am", "dich_nghia", "dich_tho"}
INPUT_GRANULARITY = "parts"  # Đọc TXT từng ảnh, xuất một TXT cho mỗi bài và mỗi bản.
RUN_NAME = f"clean_v04_{INPUT_GRANULARITY}"  # Mỗi chế độ có CSV duyệt và nhật ký riêng.
DICT_PATH = PROJECT_ROOT / "dict" / "QuocNgu_SinoNom_TongHop3.csv"

EXPORT_REVIEW = True                # Tạo bảng gợi ý local, không ghi đè bảng đã duyệt.
APPLY_REVIEW = True                # True: đọc quyết định từ REVIEW_CSV.
WRITE_OUTPUTS = True              # True: ghi TXT vào 03_text_sach của từng sách.
REQUIRE_ALL_FLAGS_REVIEWED = True   # Chặn xuất nếu còn dòng nghi vấn chưa duyệt.
REVIEW_CSV = OUTPUT_ROOT / "05_quan_ly" / RUN_NAME / "duyet_lam_sach.csv"

# Khai báo sau khi xem ảnh; chỉ khớp nguyên dòng, không xóa theo substring.
# Khóa: (mã sách, phiên bản). Ví dụ {("CPO_11", "han"): {"TIÊU ĐỀ LẶP"}}
KNOWN_NON_CONTENT_LINES = {}
# Dấu câu giữ nguyên mặc định; chỉ thêm ánh xạ đã kiểm tra theo nguồn.
PUNCTUATION_REPLACEMENTS = {}

assert INPUT_GRANULARITY in {"parts", "merged"}
assert re.fullmatch(r"[A-Za-z0-9_-]+", TOPIC_CODE)
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME)


## 2 Đọc TXT và tìm trang nguồn
`parts` đọc TXT từng ảnh của bộ `CV06_CPO_xx`. Các phần được ráp theo thứ tự số trong cùng bài và cùng phiên bản.
`merged` đọc các TXT OCR đã ghép theo từng bài.
Hai TXT dịch nghĩa Bắc Sứ được đọc trực tiếp từ `01_nguon/viet/dich_nghia` trong cả hai chế độ. Chế độ `merged` dùng cùng các TXT OCR gộp và giữ hai quyển riêng. Trang PDF Việt lấy từ `nguon_tho_viet.json`.
Trong bộ local, `05_quan_ly/ghep_theo_tac_pham.json` chỉ ra thứ tự các ảnh trong mỗi TXT gộp.
Metadata từng ảnh dùng để kiểm tra vị trí nguồn khi duyệt.


In [ ]:
# 2.1 Đọc TXT và ánh xạ về ảnh/PDF
def sha256(data):
    return hashlib.sha256(data).hexdigest()


def natural_key(text):
    return [int(s) if s.isdigit() else s.casefold() for s in re.split(r"(\d+)", text)]


def describe_source(name):
    parts = PurePosixPath(name.replace("\\", "/")).parts
    if not name.lower().endswith(".txt") or "05_quan_ly" in parts:
        return None
    stage = next((p for p in ("02_ocr_tho", "01_nguon") if p in parts), None)
    if stage is None:
        return None
    pos = parts.index(stage)
    code = next((code for code, folder in BOOK_FOLDERS.items() if folder in parts[:pos]), None)
    tail = parts[pos + 1:]
    if code not in SELECTED_CODES or not tail:
        return None
    members = {int(m.group(1)) for p in parts if (m := re.search(r"(?:^|_)TV(\d+)(?:_|$)", p))}
    if members != {4}:
        return None
    if tail[0] == "han":
        kind = "han"
    elif len(tail) > 1 and tail[0] == "viet":
        kind = tail[1]
    else:
        return None
    if kind not in SELECTED_KINDS:
        return None
    if stage == "01_nguon" and ((code, kind) != ("CPO_09", "dich_nghia") or len(tail) != 3):
        return None
    filename = parts[-1]
    merged = filename.endswith("_ocr_tho.txt")
    if stage == "02_ocr_tho" and merged != (INPUT_GRANULARITY == "merged"):
        return None
    stem = parts[-2] if stage == "02_ocr_tho" else Path(filename).stem
    return dict(code=code, kind=kind, work=stem, stage=stage)


def read_sources(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Chưa có đầu vào: {path}. Đặt INPUT_PATH đúng thư mục/ZIP OCR.")
    if path.is_dir():
        entries = [(p.relative_to(path).as_posix(), p) for p in path.rglob("*.txt")]
        get_bytes = lambda item: item.read_bytes()
        archive = None
    elif path.suffix.lower() == ".zip":
        archive = ZipFile(path)
        entries = [(p.filename, p) for p in archive.infolist() if not p.is_dir()]
        get_bytes = archive.read
    else:
        raise ValueError("INPUT_PATH phải là thư mục hoặc ZIP.")
    sources, identities = [], set()
    try:
        for name, item in sorted(entries, key=lambda pair: natural_key(pair[0])):
            info = describe_source(name)
            if info is None:
                continue
            identity = (info["code"], info["kind"], info["work"], PurePosixPath(name).name)
            if identity in identities:
                raise ValueError(f"Nguồn trùng nhiều lượt chạy: {name}. Chọn một thư mục OCR.")
            identities.add(identity)
            raw = get_bytes(item)
            sources.append(dict(info, relative_path=name, text=raw.decode("utf-8-sig"),
                                source_sha256=sha256(raw)))
    finally:
        if archive is not None:
            archive.close()
    if not sources:
        raise ValueError("Không tìm thấy TXT đúng cấu trúc. Kiểm tra INPUT_PATH và INPUT_GRANULARITY.")
    return sources


def attach_native_vietnamese_pages(source, root):
    text_path = PurePosixPath(source["relative_path"])
    book = text_path.parents[3]
    index = root / book / "05_quan_ly" / "nguon_tho_viet.json"
    if not index.is_file():
        return
    relative = text_path.relative_to(book).as_posix()
    exports = json.loads(index.read_text(encoding="utf-8"))["exports"]
    pages = next((entry["pages"] for entry in exports if entry["file"] == relative), [])
    segments = source["text"].split("\f")
    if len(segments) == len(pages) and all(
            sha256(segment.encode("utf-8")) == page["selected_sha256_utf8"]
            for segment, page in zip(segments, pages)):
        source["review_pages"] = [dict(pdf_page=page["pdf_page"]) for page in pages]


def attach_review_pages(sources, root):
    """Gắn trang theo manifest và chỉ dùng tọa độ khi TXT/ảnh khớp bản OCR."""
    root = Path(root)
    manifest = root / "05_quan_ly" / "ghep_theo_tac_pham.json"
    entries = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else []
    by_work = {(r["work"], r["sha256"]): r["source_files"]
               for r in entries if r.get("status") == "merged"}
    for source in sources:
        if source["stage"] == "01_nguon":
            attach_native_vietnamese_pages(source, root)
            continue
        text_path = PurePosixPath(source["relative_path"])
        segments = source["text"].split("\f")
        names = by_work.get((source["work"], source["source_sha256"]))
        if not text_path.name.endswith("_ocr_tho.txt") and len(segments) == 1:
            names = [text_path.with_suffix(".png").name]
        if not names or len(names) != len(segments):
            continue
        source["review_pages"] = []
        book, _, *version_folder = text_path.parent.parts
        for segment, source_name in zip(segments, names):
            image_name = PurePosixPath(source_name).name
            part = root / text_path.parent / Path(image_name).with_suffix(".txt")
            image = root / book / "01_nguon" / Path(*version_folder) / image_name
            page_match = re.search(r"_pdfp(\d+)\.png$", image_name)
            metadata = part.with_suffix(".meta.json")
            valid_image = image if image.is_file() else None
            polygons = {}
            if metadata.is_file() and part.is_file() and valid_image:
                meta = json.loads(metadata.read_text(encoding="utf-8"))
                part_bytes = part.read_bytes()
                if (sha256(part_bytes) == meta["output_hashes"]["txt"]
                    and part_bytes.decode("utf-8-sig") == segment
                    and sha256(image.read_bytes()) == meta["identity"]["source_sha256"]):
                    reading_map = meta["review"].get("reading_map", [])
                    if len(segment.split("\n")) == len(reading_map):
                        polygons = {row["txt_line"]: row["polygon"] for row in reading_map}
            source["review_pages"].append(dict(pdf_page=int(page_match[1]) if page_match else None,
                source_image=valid_image.relative_to(root).as_posix() if valid_image else "",
                polygons=polygons))


def checked_output_path(relative):
    relative = Path(relative)
    if relative.anchor or ".." in relative.parts:
        raise ValueError(f"Đường dẫn đầu ra không hợp lệ: {relative}")
    target = OUTPUT_ROOT.resolve() / relative
    if target.resolve() != target or target.is_symlink():
        raise ValueError(f"Đích đầu ra bị chuyển hướng: {target}")
    return target


In [ ]:
# 2.2 Kiểm kê file đầu vào
sources = read_sources(INPUT_PATH)
if INPUT_PATH.is_dir():
    attach_review_pages(sources, INPUT_PATH)
for key, count in sorted(Counter((s["code"], s["kind"]) for s in sources).items()):
    print(key, ":", count, "file")
print("Chế độ:", INPUT_GRANULARITY, "| Tổng:", len(sources))


## 3 Hai cell làm sạch Hán và Việt

Riêng TXT từ PDF Việt Bắc Sứ, bản sạch chuẩn hóa ký tự `ƣ` thành `ư` và bỏ mốc tờ bản Hán dạng `[53b]`. Niên đại dạng `[1759]` giữ theo nguồn.
Tự sửa khoảng trắng, ngoặc rỗng và nhãn phiên bản đứng riêng ở bản Việt phù hợp.
Chữ lẫn số, chú thích có nội dung, dòng lặp, số đứng riêng và chỗ nối qua trang cần duyệt.
Giữ tên riêng, niên hiệu, số có nghĩa, đơn vị, dấu câu, chữ Hán hiếm và `[MASK]`.
Từ điển chỉ dùng để xem tiếng dịch âm ngoài danh sách.


In [ ]:
# 3.1 Các quy tắc dùng chung
def normalize_latin(text):
    result, pending = [], []
    for ch in text:
        is_latin = "LATIN" in unicodedata.name(ch, "")
        is_mark = unicodedata.category(ch).startswith("M")
        if is_latin or (is_mark and pending):
            pending.append(ch)
        else:
            if pending:
                result.append(unicodedata.normalize("NFC", "".join(pending)))
                pending = []
            result.append(ch)
    if pending:
        result.append(unicodedata.normalize("NFC", "".join(pending)))
    return "".join(result)


def normalize_spaces(line):
    line = "".join(" " if c == "\t" or unicodedata.category(c) == "Zs" else c for c in line)
    line = re.sub(r" +", " ", line).strip(" ")
    return line


def word_spans(text):
    start = None
    for i, ch in enumerate(text + " "):
        inside = unicodedata.category(ch)[0] in "LMN"
        if inside and start is None:
            start = i
        elif not inside and start is not None:
            yield start, i, text[start:i]
            start = None


def suffix_digit_suggestion(text):
    # Chỉ gợi ý xóa hậu tố số; không sửa số đứng riêng hoặc số chen giữa chữ.
    edits = []
    for start, end, token in word_spans(text):
        split = len(token)
        while split and token[split - 1].isdigit():
            split -= 1
        prefix = token[:split]
        if split < len(token) and any(c.isalpha() for c in prefix) and not any(c.isdigit() for c in prefix):
            edits.append((start + split, end))
    for start, end in reversed(edits):
        text = text[:start] + text[end:]
    return text


def load_pronunciation_vocabulary(path):
    path = Path(path)
    if not path.is_file():
        print("Không có từ điển tại DICT_PATH; bỏ thống kê ngoài từ điển.")
        return set()
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if not reader.fieldnames or "QuocNgu" not in reader.fieldnames:
            raise ValueError("Từ điển cần cột QuocNgu.")
        return {normalize_latin(row["QuocNgu"].strip()).casefold() for row in reader}



EMPTY_BRACKETS = re.compile(r"\(\s*\)|\[\s*\]|（\s*）|【\s*】")
NOTE_MARK = re.compile(r"\(\s*\d+\s*\)|\[\s*\d+[a-z]?\s*\]|（\s*\d+\s*）|【\s*\d+\s*】")


def clean_shared(line):
    text = normalize_spaces(line)
    rules = ["whitespace"] if text != line else []
    without_empty = normalize_spaces(EMPTY_BRACKETS.sub(" ", text))
    if without_empty != text:
        rules.append("empty_brackets")
    text = without_empty
    if "[MASK]" not in text:  # Custom punctuation replacements never touch MASK.
        replaced = text
        for old, new in PUNCTUATION_REPLACEMENTS.items():
            replaced = replaced.replace(old, new)
        if replaced != text:
            rules.append("punctuation_config")
        text = replaced
    return text, rules


def common_flags(text):
    visible = text.replace("[MASK]", "")
    flags = []
    if NOTE_MARK.search(visible):
        flags.append("possible_note_marker")
    if any(visible.count(a) != visible.count(b) for a, b in
           [("(", ")"), ("[", "]"), ("（", "）"), ("【", "】")]):
        flags.append("unbalanced_brackets")
    if re.fullmatch(r"\d+", visible.strip()):
        flags.append("standalone_number")
    if any(any(c.isalpha() for c in token) and any(c.isdigit() for c in token)
           for _, _, token in word_spans(visible)):
        flags.append("letters_attached_to_digits")
    if re.search(r"[_#@=<>\\|{}~^$]", visible) or any(
            unicodedata.category(c) in {"Cc", "Cf"} or c == "\ufffd" for c in visible):
        flags.append("unexpected_symbol")
    return flags


In [ ]:
# 3.2 Làm sạch bản Hán
def clean_han(line):
    """Giữ nguyên mã chữ Hán, kể cả chữ hiếm và chữ vùng dùng riêng."""
    text, rules = clean_shared(line)
    flags = common_flags(text)
    if any(c.isalpha() and "LATIN" in unicodedata.name(c, "")
           for c in text.replace("[MASK]", "")):
        flags.append("latin_in_han")
    if re.search(r"[%％○]", text):
        flags.append("han_symbol_check")
    return text, rules, flags


In [ ]:
# 3.3 Làm sạch các bản Việt
VIET_LABELS = {
    "dich_am": {"phiên âm", "dịch âm"},
    "dich_nghia": {"dịch nghĩa"},
    "dich_tho": {"dịch thơ"},
}


def clean_viet(line, kind, *, native_pdf=False):
    nfc = normalize_latin(line)
    normalized = nfc.translate(str.maketrans("Ƣƣ", "Ưư")) if native_pdf else nfc
    without_folios = re.sub(r"\[\d{1,3}[ab]\]", "", normalized) if native_pdf else normalized
    text, rules = clean_shared(without_folios)
    if without_folios != normalized:
        rules.insert(0, "vietnamese_pdf_folio")
    if normalized != nfc:
        rules.insert(0, "vietnamese_pdf_glyph")
    if nfc != line:
        rules.insert(0, "vietnamese_nfc")
    if text.casefold().rstrip(":").rstrip() in VIET_LABELS.get(kind, set()):
        return "", rules + ["standalone_version_label"], []
    flags = common_flags(text)
    if re.search(r"[%％]", text):
        flags.append("percent_check")
    if re.fullmatch(r"chú thích\s*:?", text, re.IGNORECASE):
        flags.append("possible_note_section")
    return text, rules, flags


In [ ]:
# 3.4 Tạo dòng duyệt và gắn vị trí nguồn
def build_records(sources, vocabulary):
    records = []
    for source in sources:
        pages = source["text"].replace("\r\n", "\n").replace("\r", "\n").split("\f")
        kind = source["kind"]
        native_pdf = source["stage"] == "01_nguon"
        normalized = [[clean_han(line) if kind == "han" else clean_viet(line, kind, native_pdf=native_pdf)
                       for line in page.split("\n")] for page in pages]
        counts = Counter(text for page in normalized for text, _, _ in page if text)
        for page_no, page in enumerate(normalized, 1):
            lines = pages[page_no - 1].split("\n")
            page_maps = source.get("review_pages", [])
            location = page_maps[page_no - 1] if page_no <= len(page_maps) else {}
            last = max((i for i, (text, _, _) in enumerate(page, 1) if text), default=0)
            for line_no, (text, rules, flags) in enumerate(page, 1):
                flags = list(flags)
                if text and counts[text] > 1:
                    flags.append("repeated_line")
                if text and text in KNOWN_NON_CONTENT_LINES.get((source["code"], kind), set()):
                    flags.append("configured_non_content_line")
                if page_no < len(pages) and line_no == last:
                    flags.append("page_boundary_check")
                oov = set()
                if kind == "dich_am" and vocabulary:
                    oov = {token.casefold() for _, _, token in word_spans(text.replace("[MASK]", " "))
                           if all(unicodedata.category(c)[0] in "LM" for c in token)
                           and token.casefold() not in vocabulary}
                row_id = sha256(json.dumps([source["relative_path"], source["source_sha256"],
                                           page_no, line_no], ensure_ascii=False).encode())[:24]
                records.append(dict(row_id=row_id, relative_path=source["relative_path"],
                    source_sha256=source["source_sha256"], code=source["code"], kind=kind,
                    work=source["work"], page=page_no, line=line_no, raw=lines[line_no - 1], clean=text,
                    pdf_page=location.get("pdf_page") or "", source_image=location.get("source_image", ""),
                    polygon=location.get("polygons", {}).get(line_no),
                    automatic_rules=";".join(rules), reasons=";".join(flags),
                    suggested=suffix_digit_suggestion(text) if kind != "han" else text,
                    context_before=lines[line_no - 2] if line_no > 1 else "",
                    context_after=lines[line_no] if line_no < len(lines) else "",
                    oov_tokens=" | ".join(sorted(oov))))
    return records


## 4 Duyệt theo trang PDF và cập nhật CSV

1. Chạy cell **4.1** để có bảng `duyet_lam_sach.csv`. Cell **4.2** nạp các hàm đọc và kiểm tra bảng.
2. Khi cần tự xem ảnh, dùng cell **4.3**. Chọn `REVIEW_CODE`, `REVIEW_KIND` và `REVIEW_PDF_PAGE`, rồi bật `SHOW_FULL_PDF` nếu cần ngữ cảnh cả trang.
3. Bạn có thể duyệt trực tiếp trong CSV hoặc gửi ngay yêu cầu cho Codex. Sau khi lưu quyết định, chạy **5.1** với `APPLY_REVIEW=True` để xem kết quả.

**Prompt mẫu gửi cho Codex**

> Đọc `san_pham/TV04/05_quan_ly/clean_v04_parts/duyet_lam_sach.csv` và duyệt các dòng của `CPO_09` từ trang PDF … đến trang PDF …. Đối chiếu ảnh gốc theo `source_image`, số trang theo `pdf_page` và vùng chữ theo `polygon`. Khi cần, kiểm tra PDF, JSON OCR và metadata cùng phần. Ghi trực tiếp kết quả vào `action`, `replacement`, `note` trong CSV. Chọn `keep` nếu khớp ảnh, `replace` nếu OCR sai và ghi nguyên dòng đã sửa, hoặc `delete` nếu cả dòng là rác hay nằm ngoài nội dung chính. Nếu chưa chắc, để trống `action` và ghi lý do trong `note`. Giữ các quyết định đã có và giới hạn trong phạm vi tôi chỉ định.

Giữ các cột nguồn `row_id`, `source_sha256`, `relative_path`, `page`, `line`, `clean`. Khi đã xác nhận câu bị ngắt, có thể dùng `join_next` để nối trong cùng tác phẩm và phiên bản.


In [ ]:
# 4.1 Tạo hoặc đọc vị trí bảng duyệt
vocabulary = load_pronunciation_vocabulary(DICT_PATH)
records = build_records(sources, vocabulary)
review_rows = [dict(row, polygon=json.dumps(row["polygon"]) if row["polygon"] else "",
                    action="", replacement="", note="") for row in records]
print("Số dòng:", len(records), "| Dòng gắn cờ trước khi áp dụng CSV:", sum(bool(r["reasons"]) for r in records))
for row in records[:20]:
    print(f'{row["page"]}:{row["line"]}', repr(row["clean"]),
          "=> gợi ý:", repr(row["suggested"]), "|", row["reasons"])
print("Tiếng dịch âm ngoài từ điển (chỉ gắn cờ):",
      sorted({t for r in records for t in r["oov_tokens"].split(" | ") if t})[:50])


def csv_bytes(rows, fields):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue().encode("utf-8-sig")


def write_once(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() == data:
            return
        raise FileExistsError(f"Đã có file khác nội dung, không ghi đè: {path}")
    with path.open("xb") as stream:
        stream.write(data)


if EXPORT_REVIEW:
    checked_output_path(REVIEW_CSV.relative_to(OUTPUT_ROOT))
    if REVIEW_CSV.exists():
        print("Giữ nguyên bảng duyệt đã có:", REVIEW_CSV)
    else:
        write_once(REVIEW_CSV, csv_bytes(review_rows, list(review_rows[0])))
        print("Đã tạo:", REVIEW_CSV)


In [ ]:
# 4.2 Đọc quyết định từ CSV
def review_csv_rows(path, records):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Chưa có bảng duyệt: {path}. Chạy cell preview_review trước.")
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    expected = {r["row_id"]: r for r in records}
    if len(rows) != len(expected) or {r.get("row_id") for r in rows} != set(expected):
        raise ValueError("Bảng duyệt thiếu/trùng/thừa dòng hoặc nguồn thay đổi.")
    for row in rows:
        original = expected[row["row_id"]]
        for key in ("source_sha256", "relative_path", "clean", "page", "line"):
            if str(row.get(key, "")) != str(original[key]):
                raise ValueError(f"Bảng duyệt không khớp nguồn/cấu hình: {row['row_id']} / {key}")
    return rows


def validate_review_action(action, replacement, allowed):
    if action not in allowed:
        raise ValueError(f"Action không hợp lệ: {action}")
    if any(c in replacement for c in "\r\n\f\t"):
        raise ValueError("replacement phải là một dòng, không chứa TAB hay ngắt trang.")
    if action == "replace" and not replacement.strip():
        raise ValueError("Dùng delete thay cho replacement rỗng.")


def read_decisions(path, records):
    decisions = {}
    for row in review_csv_rows(path, records):
        action = row.get("action", "").strip().casefold()
        replacement = row.get("replacement", "")
        validate_review_action(action, replacement, {"", "keep", "replace", "delete", "join_next"})
        decisions[row["row_id"]] = dict(action=action, replacement=replacement, note=row.get("note", ""))
    return decisions


def apply_decisions(records, decisions):
    output, audit = [], []
    for record in records:
        decision = decisions.get(record["row_id"], {})
        action = decision.get("action", "")
        text = decision.get("replacement", "") if action == "replace" else record["clean"]
        deleted = action == "delete"
        output.append(dict(record, text=text, deleted=deleted, join_next=action == "join_next"))
        trace = dict(row_id=record["row_id"], source=record["relative_path"],
                     source_sha256=record["source_sha256"], page=record["page"],
                     pdf_page=record.get("pdf_page"), line=record["line"])
        if record["raw"] != record["clean"]:
            audit.append(dict(trace, before=record["raw"], after=record["clean"],
                action="automatic", rules=record["automatic_rules"]))
        if action:
            audit.append(dict(trace, before=record["clean"], after=None if deleted else text,
                action=action, note=decision.get("note", "")))
    return output, audit


def assemble_works(rows):
    groups = defaultdict(list)
    for row in rows:
        groups[(row["code"], row["kind"], row["work"])].append(row)
    result = {}
    for key, group in groups.items():
        chunks = []
        pending = False
        previous_location = None
        has_text_on_part = False
        for row in group:
            location = (row["relative_path"], row["page"])
            if location != previous_location:
                if previous_location is not None and not pending:
                    chunks.append("\f")
                has_text_on_part = False
                previous_location = location
            if row["deleted"] or not row["text"]:
                continue
            if pending:
                chunks.append("" if key[1] == "han" else " ")
            elif has_text_on_part:
                chunks.append("\n")
            chunks.append(row["text"])
            pending = row["join_next"]
            has_text_on_part = True
        if pending:
            raise ValueError(f"join_next ở cuối tác phẩm không có dòng tiếp: {key}")
        result[key] = "".join(chunks)
    return result


In [ ]:
# 4.3 Chọn trang để đối chiếu
REVIEW_CODE = "CPO_09"
REVIEW_KIND = "han"  # Chọn "dich_nghia" để xem PDF Việt Bắc Sứ.
REVIEW_PDF_PAGE = None  # Ví dụ 49. Để None để liệt kê các trang có cảnh báo.
SHOW_FULL_PDF = False  # True để hiện cả trang PDF, cần pypdfium2.


def flagged_pages(records, kind):
    groups = defaultdict(list)
    for row in records:
        if row["reasons"] and row["kind"] == kind:
            groups[(row["code"], row["pdf_page"])].append(row)
    return groups


def image_data_uri(image, limit):
    import base64
    image.thumbnail(limit)
    stream = io.BytesIO()
    image.convert("RGB").save(stream, format="PNG")
    return "data:image/png;base64," + base64.b64encode(stream.getvalue()).decode("ascii")


def pdf_page_image(code, page, kind):
    from contextlib import closing
    import runpy
    import pypdfium2 as pdfium

    key = {"CPO_09": f"bac_su_{'han' if kind == 'han' else 'viet'}",
           "CPO_11": "van_sach", "CPO_12": "ly_tran"}[code]
    sources_by_book = runpy.run_path(str(PROJECT_ROOT / "ten_file.py"))["TEN_NGUON"]
    with pdfium.PdfDocument(sources_by_book[key]) as document:
        if not 1 <= page <= len(document):
            raise ValueError(f"Trang PDF nằm ngoài phạm vi: {page}")
        with closing(document[page - 1]) as pdf_page:
            with closing(pdf_page.render(scale=2)) as bitmap:
                return bitmap.to_pil().copy()


def review_image(row):
    from PIL import Image, ImageDraw

    if not row["source_image"]:
        return None
    with Image.open(INPUT_PATH / row["source_image"]) as original:
        polygon = row["polygon"]
        if not polygon:
            return image_data_uri(original.copy(), (650, 680))
        points = [tuple(point) for point in polygon]
        xs, ys = zip(*points)
        pad_x = max(12, int((max(xs) - min(xs)) * 0.25))
        pad_y = max(12, int((max(ys) - min(ys)) * 0.25))
        left, top = max(0, int(min(xs)) - pad_x), max(0, int(min(ys)) - pad_y)
        right = min(original.width, int(max(xs)) + pad_x)
        bottom = min(original.height, int(max(ys)) + pad_y)
        if right <= left or bottom <= top:
            return image_data_uri(original.copy(), (650, 680))
        region = original.crop((left, top, right, bottom)).convert("RGB")
        border = [(x - left, y - top) for x, y in points]
        ImageDraw.Draw(region).line(border + [border[0]], fill="#b52424", width=3)
        return image_data_uri(region, (650, 680))


def show_review_page(code, page, kind, full_pdf=False):
    from html import escape
    from IPython.display import HTML, display

    page_rows = flagged_pages(records, kind).get((code, page), [])
    if not page_rows:
        print("Trang này không có cảnh báo trong phạm vi đã chọn.")
        return
    if full_pdf or any(not row["source_image"] for row in page_rows):
        try:
            image = image_data_uri(pdf_page_image(code, page, kind), (1700, 2000))
            display(HTML(f'<img src="{image}" alt="Trang PDF nguon" style="max-width:100%">'))
        except ImportError:
            print("Muốn hiện cả trang PDF, cài pypdfium2 trong kernel đang chạy.")
        except (OSError, ValueError, IndexError, RuntimeError) as exc:
            print(f"Không mở được trang PDF tương ứng: {exc}")

    saved = {}
    if REVIEW_CSV.is_file():
        saved = {row["row_id"]: row for row in review_csv_rows(REVIEW_CSV, records)}
    table = ['<table><tr><th>Ảnh nguồn</th><th>TXT và quyết định</th></tr>']
    for row in page_rows:
        image = review_image(row)
        preview = f'<img src="{image}" alt="Vùng chữ">' if image else "Xem cả trang PDF"
        state = saved.get(row["row_id"], {})
        detail = "\n".join([
            f'{row["kind"]} · {row["work"]} · dòng {row["line"]}',
            row["row_id"], row["reasons"], "",
            f'Trước    {row["context_before"]}',
            f'OCR      {row["raw"]}',
            f'Sau      {row["context_after"]}',
            f'Tự sửa   {row["clean"]}',
            f'Quyết định {state.get("action", "")} {state.get("replacement", "")}',
            f'Ghi chú  {state.get("note", "")}',
        ])
        table.append(f'<tr><td>{preview}</td><td style="text-align:left">'
                     f'<pre style="white-space:pre-wrap">{escape(detail)}</pre></td></tr>')
    display(HTML("".join(table) + "</table>"))

if REVIEW_PDF_PAGE is None:
    pages = [(page, len(rows)) for (code, page), rows in flagged_pages(records, REVIEW_KIND).items()
             if code == REVIEW_CODE and page is not None]
    print(f"Trang PDF có cảnh báo của {REVIEW_CODE} (trang, số dòng):", sorted(pages))
    unknown = sum(len(rows) for (code, page), rows in flagged_pages(records, REVIEW_KIND).items() if page is None)
    if unknown:
        print(f"{unknown} dòng thiếu ánh xạ trang; đối chiếu thủ công trước khi chốt CSV.")
else:
    show_review_page(REVIEW_CODE, REVIEW_PDF_PAGE, REVIEW_KIND, SHOW_FULL_PDF)


## 5 Xem trước và xuất TXT sạch
Đặt `APPLY_REVIEW=True` và chạy đến cell **5.1** để xem lại quyết định.
Khi đã duyệt xong, đặt `WRITE_OUTPUTS=True` và chạy lại đến **5.2**.

Trong `parts`, mỗi bài và phiên bản có một TXT. Cấu trúc ví dụ cho `CV06_CPO_12`

```text
san_pham/TV04/CV06_CPO_12/03_text_sach/
├── han/
│   └── <mã bài>_han_v01.txt
└── viet/
    ├── dich_am/    # Phiên âm theo bài
    ├── dich_nghia/ # Dịch nghĩa theo bài
    └── dich_tho/   # Dịch thơ theo bài
```

Notebook tạo thư mục cho các bản có dữ liệu. CSV duyệt, nhật ký và manifest nằm trong
`san_pham/TV04/05_quan_ly/<RUN_NAME>/`. Nếu một TXT đích đã có nội dung khác, notebook
báo tên file và dừng để bạn kiểm tra bản đang có.


In [ ]:
# 5.1 Xem trước khi xuất
decisions = read_decisions(REVIEW_CSV, records) if APPLY_REVIEW else {}
unresolved = [r for r in records if r["reasons"] and not decisions.get(r["row_id"], {}).get("action")]
cleaned_rows, change_log = apply_decisions(records, decisions)
cleaned_works = assemble_works(cleaned_rows)
print("Chưa duyệt:", len(unresolved), "| Thay đổi có log:", len(change_log))
for key, text in list(cleaned_works.items())[:2]:
    print("\nTÁC PHẨM:", key)
    print(text[:2000])


In [ ]:
# 5.2 Xuất TXT và nhật ký
def safe_component(name):
    if name in {"", ".", ".."} or any(c in name for c in '/\\:*?"<>|\r\n\t'):
        raise ValueError(f"Tên đầu ra không an toàn: {name!r}")
    return name


TEXT_FOLDERS = {
    "han": Path("han"),
    "dich_am": Path("viet") / "dich_am",
    "dich_nghia": Path("viet") / "dich_nghia",
    "dich_tho": Path("viet") / "dich_tho",
}


def clean_version_folder(code, kind):
    if not re.fullmatch(r"CPO_\d{2}", code):
        raise ValueError(f"Mã sách không hợp lệ: {code}")
    return Path(f"{TOPIC_CODE}_{code}") / "03_text_sach" / TEXT_FOLDERS[kind]


def make_export_files(works, rows, audit, sources):
    files, manifest = {}, []
    book_texts = defaultdict(list)
    for (code, kind, stem), text in sorted(works.items(), key=lambda kv: natural_key("/".join(kv[0]))):
        stem = safe_component(stem)
        relative = clean_version_folder(code, kind) / (stem + ".txt")
        files[relative] = text.encode("utf-8")
        if INPUT_GRANULARITY == "merged":
            book_texts[(code, kind)].append((stem, text))
        source_rows = [r for r in rows if (r["code"], r["kind"], r["work"]) == (code, kind, stem)]
        manifest.append(dict(code=code, kind=kind, work=stem, file=relative.as_posix(),
            sha256=sha256(files[relative]), source_rows=[
                dict(row_id=r["row_id"], source=r["relative_path"], source_sha256=r["source_sha256"],
                     page=r["page"], line=r["line"], deleted=r["deleted"], join_next=r["join_next"])
                for r in source_rows]))
    for (code, kind), parts in book_texts.items():
        filename = "han.txt" if kind == "han" else f"viet_{kind}.txt"
        files[clean_version_folder(code, kind) / filename] = "\f".join(text for _, text in parts).encode("utf-8")
    text_outputs = [dict(file=path.as_posix(), sha256=sha256(data)) for path, data in files.items()]
    management = Path("05_quan_ly") / RUN_NAME
    encode_json = lambda data: (json.dumps(data, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
    files[management / "work_manifest.json"] = encode_json(manifest)
    files[management / "changes.jsonl"] = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in audit).encode("utf-8")
    files[management / "summary.json"] = encode_json(dict(
        stage="step_5_reviewed_draft_not_alignment_or_quota_certification",
        source_files=len(sources), works=len(works), changed_or_reviewed_lines=len(audit),
        input_granularity=INPUT_GRANULARITY, unresolved_lines=len(unresolved),
        punctuation_replacements=PUNCTUATION_REPLACEMENTS, text_outputs=text_outputs,
        sources=[dict(path=s["relative_path"], sha256=s["source_sha256"]) for s in sources]))
    return files


if WRITE_OUTPUTS:
    if REQUIRE_ALL_FLAGS_REVIEWED and unresolved:
        raise ValueError(f"Còn {len(unresolved)} dòng cần duyệt. Chưa xuất text sạch.")
    if INPUT_PATH.is_dir():
        for source in sources:
            path = INPUT_PATH / source["relative_path"]
            if sha256(path.read_bytes()) != source["source_sha256"]:
                raise ValueError(f"TXT nguồn đã thay đổi: {path}. Đọc và duyệt lại.")
    if APPLY_REVIEW and read_decisions(REVIEW_CSV, records) != decisions:
        raise ValueError("CSV đã thay đổi sau khi xem trước. Chạy lại apply_preview.")
    files = make_export_files(cleaned_works, cleaned_rows, change_log, sources)
    # Kiểm tra tất cả đích trước khi ghi để tránh xuất nửa bộ vì xung đột sẵn có.
    for relative, data in files.items():
        target = checked_output_path(relative)
        if target.exists() and target.read_bytes() != data:
            raise FileExistsError(f"File đích đã khác nội dung: {target}. Kiểm tra file hiện có.")
    for relative, data in files.items():
        write_once(checked_output_path(relative), data)
    print("Đã lưu TXT sạch vào 03_text_sach của từng sách dưới:", OUTPUT_ROOT)
    print("Nhật ký:", REVIEW_CSV.parent)
else:
    print("WRITE_OUTPUTS=False: chỉ xem trước, chưa ghi text sạch.")


## 6 Dọn bản xuất để chạy lại
Cell bên dưới được comment toàn bộ. Sau khi bỏ comment, chạy với
`DELETE_OUTPUTS=False` để xem danh sách. Chuyển sang `True` khi đã kiểm tra.

Cell lấy danh sách TXT đã xuất từ `summary.json` của lượt hiện tại và kiểm tra hash
trước khi dọn. CSV duyệt, TXT do bạn thêm hoặc sửa và dữ liệu OCR nguồn được giữ lại.


In [ ]:
DELETE_OUTPUTS = False  # True để xóa; mặc định chỉ xem trước.
management = Path("05_quan_ly") / RUN_NAME
summary_path = checked_output_path(management / "summary.json")

if not summary_path.is_file():
    print("Lượt này chưa có danh sách file đã xuất:", summary_path)
else:
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    allowed = {clean_version_folder(code, kind)
               for code in SELECTED_CODES for kind in TEXT_FOLDERS}
    targets = []
    for item in summary["text_outputs"]:
        relative = Path(item["file"])
        if relative.parent not in allowed or relative.suffix.lower() != ".txt":
            raise ValueError(f"Không thuộc các thư mục TXT sạch đã chọn: {relative}")
        path = checked_output_path(relative)
        if path.exists():
            if not path.is_file() or sha256(path.read_bytes()) != item["sha256"]:
                raise ValueError(f"File đã thay đổi sau khi xuất: {path}")
            targets.append(path)

    for name in ("work_manifest.json", "changes.jsonl", "summary.json"):
        path = checked_output_path(management / name)
        if path.is_file():
            targets.append(path)
    for path in targets:
        print(path)
    if DELETE_OUTPUTS:
        for path in targets:
            path.unlink()
        print(f"Đã xóa {len(targets)} file đã xuất.")
    else:
        print(f"Xem trước {len(targets)} file. Đổi DELETE_OUTPUTS=True để xóa.")
